In [ ]:
# =============================================================================
# ETA Prediction — Step 6: Modeling Comparison (Multiple Models)
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import lightgbm as lgb
import xgboost as xgb

print(" Imports ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Imports ready.


In [ ]:
# ====================== 6.1 LOAD DATA ======================
CKPT_DIR = "/content/drive/MyDrive/LaDe/Checkpoints_v3"   # đổi sang checkpoint mới, no-leakage

X_train = joblib.load(f"{CKPT_DIR}/X_train.pkl")
X_val   = joblib.load(f"{CKPT_DIR}/X_val.pkl")
X_test  = joblib.load(f"{CKPT_DIR}/X_test.pkl")

y_train = joblib.load(f"{CKPT_DIR}/y_train.pkl")
y_val   = joblib.load(f"{CKPT_DIR}/y_val.pkl")
y_test  = joblib.load(f"{CKPT_DIR}/y_test.pkl")

y_train_min = joblib.load(f"{CKPT_DIR}/y_train_min.pkl")
y_val_min   = joblib.load(f"{CKPT_DIR}/y_val_min.pkl")
y_test_min  = joblib.load(f"{CKPT_DIR}/y_test_min.pkl")

final_features = joblib.load(f"{CKPT_DIR}/final_features.pkl")

print(f"Data loaded - Features: {len(final_features)}")
print(final_features)   # in ra để tự kiểm tra processing_time đã biến mất chưa

Data loaded - Features: 24
['lat', 'lng', 'distance_km', 'accept_minute', 'day_of_week', 'hour_sin', 'hour_cos', 'time_to_window_start', 'time_window_duration', 'aoi_frequency', 'courier_frequency', 'distance_x_hour', 'courier_eta_mean', 'courier_eta_std', 'courier_dist_mean', 'region_id_te', 'aoi_id_te', 'aoi_type_te', 'is_weekend_te', 'is_morning_rush_te', 'is_evening_rush_te', 'is_long_haul_te', 'gps_both_te', 'coord_cluster_v2_te']


In [ ]:
# ====================== 6.1b LEAKAGE SANITY CHECK ======================
leaky_keywords = ['processing_time', 'processing_x_rush', 'courier_proc_mean']
found_leaky = [f for f in final_features if any(k in f for k in leaky_keywords)]

if found_leaky:
    raise ValueError(f"⚠️ Phát hiện feature leak vẫn còn trong final_features: {found_leaky}")
else:
    print("✅ Không phát hiện feature leak đã biết.")

✅ Không phát hiện feature leak đã biết.


In [ ]:
# ====================== 6.2 EVALUATION FUNCTION ======================
def evaluate(y_true_log, y_pred_log, y_true_min, name="Model"):
    y_pred_min = np.expm1(y_pred_log)
    mae = mean_absolute_error(y_true_min, y_pred_min)
    rmse = np.sqrt(mean_squared_error(y_true_min, y_pred_min))
    mape = np.mean(np.abs((y_true_min - y_pred_min) / (y_true_min + 1))) * 100   # avoid div0

    print(f"\n📊 {name:25s} | MAE: {mae:6.2f} mins | RMSE: {rmse:6.2f} | MAPE: {mape:5.2f}%")
    return mae, rmse, mape

results = []

In [ ]:
# ====================== 6.3 SPEED HEURISTIC ======================
# Giả sử tốc độ trung bình của courier (km/h)
avg_speed_kmh = 15.0   # có thể tuning sau

X_test_dist = X_test['distance_km'].values
speed_pred_log = np.log1p(X_test_dist * 60 / avg_speed_kmh)   # convert to minutes
evaluate(y_test, speed_pred_log, y_test_min, "SPEED Heuristic (15km/h)")


📊 SPEED Heuristic (15km/h)  | MAE: 169.69 mins | RMSE: 289.21 | MAPE: 93.36%


(169.68919198798673,
 np.float64(289.2070932419365),
 np.float64(93.35875225930236))

In [ ]:
# ====================== 6.4 CLASSICAL MODELS (Fixed: scaling + outlier clip) ======================
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("\n=== Training Classical Models (with Imputation + Scaling) ===")

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

# 6.4: Ridge -> RidgeCV
from sklearn.linear_model import RidgeCV

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

lr_pipeline = Pipeline([
    ('imputer', imputer),
    ('scaler', scaler),
    ('model', RidgeCV(alphas=np.logspace(-2, 5, 30)))
])
lr_pipeline.fit(X_train, y_train)
pred_lr = lr_pipeline.predict(X_test)
print("Best alpha chosen:", lr_pipeline.named_steps['model'].alpha_)
print("Pred log range:", pred_lr.min(), pred_lr.max())
evaluate(y_test, pred_lr, y_test_min, "Linear Regression (RidgeCV)")

# Re-check condition number sau khi giảm cat_cols
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)
print(f"Condition number (after fix): {np.linalg.cond(X_train_scaled):.2e}")

# 2. KNN — cũng cần scale (distance-based)
knn_pipeline = Pipeline([
    ('imputer', imputer),
    ('scaler', scaler),
    ('model', KNeighborsRegressor(n_neighbors=50, weights='distance', n_jobs=-1))
])
knn_pipeline.fit(X_train, y_train)
pred_knn = knn_pipeline.predict(X_test)
evaluate(y_test, pred_knn, y_test_min, "KNN Regressor")

# 3. Decision Tree — KHÔNG cần scale (tree-based, threshold-split không quan tâm scale)
dt_pipeline = Pipeline([
    ('imputer', imputer),
    ('model', DecisionTreeRegressor(max_depth=12, random_state=42))
])
dt_pipeline.fit(X_train, y_train)
pred_dt = dt_pipeline.predict(X_test)
evaluate(y_test, pred_dt, y_test_min, "Decision Tree")


=== Training Classical Models (with Imputation + Scaling) ===
Best alpha chosen: 1172.1022975334818
Pred log range: 2.4182919744274702 15.900973615520538

📊 Linear Regression (RidgeCV) | MAE: 203.78 mins | RMSE: 23379.69 | MAPE: 154.72%
Condition number (after fix): 1.24e+01

📊 KNN Regressor             | MAE:  69.20 mins | RMSE: 116.22 | MAPE: 91.22%

📊 Decision Tree             | MAE:  51.12 mins | RMSE:  88.78 | MAPE: 80.95%


(51.11597589495168,
 np.float64(88.77748566653301),
 np.float64(80.94632668934848))

In [ ]:
# ====================== 6.5 GRADIENT BOOSTING ======================

# 4. LightGBM — tăng num_boost_round, để early stopping tự quyết định điểm dừng
print("\nTraining LightGBM...")
lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 128,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'verbose': -1,
    'random_state': 42
}

train_set = lgb.Dataset(X_train, label=y_train)
val_set = lgb.Dataset(X_val, label=y_val, reference=train_set)

lgb_model = lgb.train(
    lgb_params, train_set,
    num_boost_round=5000,   # đặt cao, để early stopping tự cắt
    valid_sets=[val_set],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)
print(f"LightGBM stopped at iteration: {lgb_model.best_iteration}")

pred_lgb = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
evaluate(y_test, pred_lgb, y_test_min, "LightGBM")

# 5. XGBoost — thêm early_stopping_rounds
print("\nTraining XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=5000,            # đặt cao, để early stopping tự cắt
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:absoluteerror',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=100     # thêm dòng này
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
print(f"XGBoost stopped at iteration: {xgb_model.best_iteration}")

pred_xgb = xgb_model.predict(X_test, iteration_range=(0, xgb_model.best_iteration + 1))
evaluate(y_test, pred_xgb, y_test_min, "XGBoost")


Training LightGBM...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 0.469701
[200]	valid_0's l1: 0.461882
[300]	valid_0's l1: 0.458805
[400]	valid_0's l1: 0.456504
[500]	valid_0's l1: 0.455081
[600]	valid_0's l1: 0.453791
[700]	valid_0's l1: 0.452643
[800]	valid_0's l1: 0.451498
[900]	valid_0's l1: 0.450657
[1000]	valid_0's l1: 0.450032
[1100]	valid_0's l1: 0.449541
[1200]	valid_0's l1: 0.449092
[1300]	valid_0's l1: 0.448402
[1400]	valid_0's l1: 0.447914
[1500]	valid_0's l1: 0.447477
[1600]	valid_0's l1: 0.447033
[1700]	valid_0's l1: 0.446772
[1800]	valid_0's l1: 0.446491
[1900]	valid_0's l1: 0.446216
[2000]	valid_0's l1: 0.445958
[2100]	valid_0's l1: 0.445775
[2200]	valid_0's l1: 0.445577
[2300]	valid_0's l1: 0.445385
[2400]	valid_0's l1: 0.445187
[2500]	valid_0's l1: 0.44507
[2600]	valid_0's l1: 0.444934
[2700]	valid_0's l1: 0.444863
[2800]	valid_0's l1: 0.444673
[2900]	valid_0's l1: 0.444536
[3000]	valid_0's l1: 0.444464
[3100]	valid_0's l1: 0.444

(43.981010819540955,
 np.float64(84.22080046059638),
 np.float64(82.74848196466495))

In [ ]:
# ====================== 6.6 SAVE BEST MODEL ======================
joblib.dump(lgb_model, f"{CKPT_DIR}/best_lgb_model_v2.pkl")
joblib.dump(xgb_model, f"{CKPT_DIR}/xgb_model_v2.pkl")

print(f"\n✅ Best models saved in {CKPT_DIR}")


✅ Best models saved in /content/drive/MyDrive/LaDe/Checkpoints_v3


In [ ]:
from sklearn.linear_model import LinearRegression

def compute_vif(X_scaled_df):
    vif_data = []
    for i, col in enumerate(X_scaled_df.columns):
        y_col = X_scaled_df[col]
        X_others = X_scaled_df.drop(columns=[col])
        r2 = LinearRegression().fit(X_others, y_col).score(X_others, y_col)
        vif = 1 / (1 - r2) if r2 < 0.9999 else np.inf
        vif_data.append((col, vif))
    return pd.DataFrame(vif_data, columns=['feature', 'VIF']).sort_values('VIF', ascending=False)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
vif_df = compute_vif(X_train_scaled_df)
print(vif_df.head(15))

                feature        VIF
2           distance_km  25.010262
11      distance_x_hour  23.419402
19   is_morning_rush_te   6.431818
5              hour_sin   5.642157
6              hour_cos   3.664505
12     courier_eta_mean   2.808758
4           day_of_week   2.647056
18        is_weekend_te   2.645326
23  coord_cluster_v2_te   2.540972
13      courier_eta_std   2.496616
15         region_id_te   2.459579
20   is_evening_rush_te   1.988508
1                   lng   1.514519
16            aoi_id_te   1.501389
21      is_long_haul_te   1.318862
